# CallWhisper-8k: Decoding Adaptation Sweeps

Purpose: run cheap, non-training adaptation before considering LoRA. These experiments test decoding choices on the same fixed manifest.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = ""  # Optional: set after pushing, e.g. https://github.com/<user>/callwhisper-8k.git
REPO_DIR = Path("/content/callwhisper-8k")

if not REPO_DIR.exists():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL, or upload/clone the repo to /content/callwhisper-8k first.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo:", Path.cwd())

In [ ]:
!nvidia-smi
!python -m pip install -U pip
!python -m pip install -e ".[dev]"
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

expected_audio_dir = REPO_DIR / "datasets/GV_Dev_5h/Audio"
drive_audio_dir = Path("/content/drive/MyDrive/callwhisper-8k/datasets/GV_Dev_5h/Audio")

if not expected_audio_dir.exists() and drive_audio_dir.exists():
    expected_audio_dir.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(drive_audio_dir, expected_audio_dir)

print("Audio exists:", expected_audio_dir.exists())

## Sweep Config

Use `small` for fast iteration. Switch `MODEL` to `medium` after the sweep shape is working.

In [ ]:
MODEL = "small"
MANIFEST = "datasets/manifests/gramvaani_dev_50.csv"

sweeps = [
    {"name": "baseline_manifest", "args": ["--language-mode", "manifest"]},
    {"name": "language_auto", "args": ["--language-mode", "auto"]},
    {"name": "language_hi", "args": ["--language-mode", "hi"]},
    {"name": "beam_1", "args": ["--language-mode", "manifest", "--beam-size", "1"]},
    {"name": "beam_5", "args": ["--language-mode", "manifest", "--beam-size", "5"]},
    {"name": "temperature_0", "args": ["--language-mode", "manifest", "--temperature", "0"]},
    {"name": "temperature_02", "args": ["--language-mode", "manifest", "--temperature", "0.2"]},
    {"name": "no_previous_text", "args": ["--language-mode", "manifest", "--condition-on-previous-text", "false"]},
    {"name": "hindi_prompt", "args": ["--language-mode", "manifest", "--initial-prompt", "यह हिंदी फोन कॉल ऑडियो है।"]},
]

In [ ]:
for sweep in sweeps:
    output_json = f"results/colab_adaptation_{MODEL}_{sweep['name']}_seed0.json"
    cmd = [
        sys.executable, "-m", "callwhisper.eval",
        "--manifest", MANIFEST,
        "--model", MODEL,
        "--seed", "0",
        "--output-json", output_json,
        *sweep["args"],
    ]
    print("RUN", " ".join(cmd))
    subprocess.run(cmd, check=True, env={**os.environ, "PYTHONPATH": "src"})

In [ ]:
import json, pandas as pd

rows = []
for path in sorted(Path("results").glob(f"colab_adaptation_{MODEL}_*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "experiment": path.stem.replace(f"colab_adaptation_{MODEL}_", ""),
        "files": payload["summary"]["num_files"],
        "wer": round(payload["summary"]["wer"], 4),
        "cer": round(payload["summary"]["cer"], 4),
    })

df = pd.DataFrame(rows).sort_values("wer")
df

In [ ]:
Path(f"results/colab_adaptation_{MODEL}_summary.md").write_text(df.to_markdown(index=False) + "\n", encoding="utf-8")
print(Path(f"results/colab_adaptation_{MODEL}_summary.md").read_text(encoding="utf-8"))